In [ ]:
'/mnt/OPERA3/'

In [2]:
from pathlib import Path

# --- CONFIGURATION ---

# 1. MAPPING: Target Mouse ID (int) -> Unique Substring in VSI Filename (str)
# This mapping is used to match the VSI file to its correct destination mouse_i folder.
MOUSE_MAPPING = {
    12: '12b',
    13: '5693',
    14: '5692',
    15: 'Mouse15',
    16: 'Mouse16'
}

# 2. SOURCE DIRECTORY: Full, local mount path on your Ubuntu desktop for the cloud storage.
# !!! REPLACE THE PLACEHOLDER BELOW with the actual path where your cloud share is mounted on Ubuntu !!!
SOURCE_BASE_DIR = Path("/mnt/NEMO/home/shared/Shared - Baptiste Pradel/Histology/rep2")

# 3. DESTINATION DIRECTORY: The local path on your Ubuntu desktop for the Synology server mount.
# !!! REPLACE THE PLACEHOLDER BELOW with the actual path where the Synology server is mounted on Ubuntu !!!
DEST_BASE_DIR = Path("/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/rep2")


# --- RSYNC COMMAND GENERATION ---

rsync_commands = []
found_files = set()

print("\n--- Generating Rsync Commands ---")

# --- DYNAMIC FILE DISCOVERY ---
required_substrings = list(MOUSE_MAPPING.values())

try:
    # Use glob to find all actual VSI files in the source directory
    vsi_paths = list(SOURCE_BASE_DIR.glob('*.vsi'))
    
    # Filter: Only keep file names that contain one of the required substrings
    VSI_FILE_LIST = [
        p.name for p in vsi_paths 
        if p.is_file() and any(sub in p.name for sub in required_substrings)
    ]
    
    print(f"Found {len(vsi_paths)} total .vsi files. Filtered to {len(VSI_FILE_LIST)} matching the mouse mappings.")

except Exception as e:
    print(f"Error accessing source directory ({SOURCE_BASE_DIR}): {e}. Check mount point and permissions.")
    VSI_FILE_LIST = [] # Fail gracefully if the directory cannot be accessed

# -------------------------------

for mouse_num, substring in MOUSE_MAPPING.items():
    
    # 1. Determine the destination directory
    # e.g., /.../rep2/mouse_12/vsi
    dest_path = DEST_BASE_DIR / f"mouse_{mouse_num}" / "vsi"
    
    # Ensure the mouse directory exists in the destination path
    if not dest_path.exists():
        # This will create the required mouse_i/vsi directory if it was not created
        dest_path.mkdir(parents=True, exist_ok=True) 

    # 2. Find the matching VSI filename
    matching_vsi_fn = None
    for filename in VSI_FILE_LIST:
        if substring in filename:
            # Found the match for this mouse
            matching_vsi_fn = filename
            found_files.add(filename)
            break
            
    if matching_vsi_fn:
        # 3. Construct the full source paths
        vsi_file_source = SOURCE_BASE_DIR / matching_vsi_fn
        
        # 3a. Strip the .vsi extension using Path().stem
        stripped_fn = Path(matching_vsi_fn).stem
        
        # The accompanying folder name is the VSI filename (stripped) enclosed in underscores
        vsi_folder_name = f"_{stripped_fn}_"
        vsi_folder_source = SOURCE_BASE_DIR / vsi_folder_name
        
        # rsync destination is the vsi directory itself, ensuring the files/folders land inside it.
        dest_target = f"{dest_path}/"
        
        # 4. Generate the robust rsync commands (two commands per mouse)
        
        # Command 1: Transfer the VSI file
        vsi_cmd = (
            f'rsync -avz --progress "{vsi_file_source}" "{dest_target}"'
        )
        rsync_commands.append(vsi_cmd)
        
        # Command 2: Transfer the VSI accompanying folder
        vsi_folder_cmd = (
            f'rsync -avz --progress "{vsi_folder_source}" "{dest_target}"'
        )
        rsync_commands.append(vsi_folder_cmd)
        
        print(f"Match found for mouse_{mouse_num} ({substring}): Generated commands for VSI file and accompanying folder.")
    else:
        print(f"⚠️ Warning: No VSI file containing substring '{substring}' found in VSI_FILE_LIST for mouse_{mouse_num}.")

print("\n--- Rsync Script for Sequential Transfer ---")
print("# Run these commands one at a time for maximum stability and resumability.")
print("# Each mouse requires two separate rsync commands: one for the .vsi file and one for the corresponding folder.")
print("\n# --- START RSYNC COMMANDS ---")

for cmd in rsync_commands:
    print(cmd)

print("# --- END RSYNC COMMANDS ---")

print(f"\n✅ Total transfer operations queued: {len(rsync_commands)}.")


--- Generating Rsync Commands ---
Found 18 total .vsi files. Filtered to 5 matching the mouse mappings.
Match found for mouse_12 (12b): Generated commands for VSI file and accompanying folder.
Match found for mouse_13 (5693): Generated commands for VSI file and accompanying folder.
Match found for mouse_14 (5692): Generated commands for VSI file and accompanying folder.
Match found for mouse_15 (Mouse15): Generated commands for VSI file and accompanying folder.
Match found for mouse_16 (Mouse16): Generated commands for VSI file and accompanying folder.

--- Rsync Script for Sequential Transfer ---
# Run these commands one at a time for maximum stability and resumability.
# Each mouse requires two separate rsync commands: one for the .vsi file and one for the corresponding folder.

# --- START RSYNC COMMANDS ---
rsync -avz --progress "/mnt/NEMO/home/shared/Shared - Baptiste Pradel/Histology/rep2/20251109_40X_Mouse12b_Timer_PZA-RIF_20251110_11.vsi" "/mnt/OPERA3/Nathan/data/macrohet/mtb_